# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaaDasim05/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen lane is **Refresh / Content Opportunity Scoring**.

I frame this as a **scoring/ranking problem**. The goal is to assign each content item an opportunity score indicating how strongly it appears to need a refresh.

The model would rank content items from higher-priority refresh opportunities to lower-priority opportunities.

The output is not intended to prove that a page must be refreshed. It is a decision-support signal that helps a content team decide which pages deserve review first.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no direct ground-truth column in the starter dataset saying whether a page genuinely needs a refresh.

Therefore, I would initially use an observable performance-based proxy.

One candidate proxy is whether impressions declined by more than 20% compared with the previous 30-day period:

impressions_last_30d < 0.8 × impressions_prev_30d

This proxy represents a measurable deterioration in search visibility. I would combine this with other contextual features such as content age, days since last update, CTR, and average position rather than treating the proxy as proof that a refresh is required.

The proxy should therefore be interpreted as a signal for prioritization, not as a ground-truth business decision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric would be **Precision@K**, with Precision@50 as an initial example.

The content team has limited time, so the important question is not whether the model classifies every page correctly. The important question is whether the top-ranked pages contain a high proportion of genuine opportunity signals.

Precision@50 measures the proportion of the 50 highest-ranked pages that meet the selected deterioration proxy.

A secondary metric could be recall, but Precision@K is more closely aligned with the practical decision of creating a small prioritized review queue.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/SaaDasim05/Flyrank-ML-Internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
df[[
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_last_30d",
    "impressions_prev_30d",
    "ctr",
    "avg_position",
    "trend_direction"
]].head(10)

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_last_30d,impressions_prev_30d,ctr,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,578,987,0.76,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,2501,5915,0.05,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,2382,6089,0.09,36.5,down
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,3626,4206,0.49,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,4211,6452,0.13,44.0,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,617,1009,0.03,8.5,down
6,content_9a34b442b552,client_8722616204,keyword article,90,20,1,13,0.00,7.0,down
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,636,632,0.06,21.2,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,5696,13828,0.09,46.0,down
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,252,356,0.16,4.9,down


The unit of analysis is **one content item/page**.

Each row represents one content item belonging to a client. The features describe that item's search performance, age, freshness, engagement, and trend.

For example, the dataframe contains one row per content item with fields such as impressions, CTR, average position, content age, and days since the last update.

The eventual model would assign each content item an opportunity score, allowing the pages to be ranked for review.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as "refresh every page older than 180 days" is easy to understand, but it treats all pages with the same age as equally important.

The data suggests that content performance depends on multiple signals, including impressions, CTR, average position, content age, freshness, and trends.

An ML scoring approach can combine these signals and produce a ranked list instead of applying the same decision to every page.

However, ML is only useful if it improves the prioritization compared with a reasonable fixed baseline. Therefore, the eventual model should be compared against simple rules rather than assuming that ML is automatically better.

In [10]:
df["refresh_proxy"] = (
    df["impressions_last_30d"] <
    0.8 * df["impressions_prev_30d"]
).astype(int)

print(df["refresh_proxy"].value_counts())
print("Proxy rate:", df["refresh_proxy"].mean())

refresh_proxy
1    16262
0    13738
Name: count, dtype: int64
Proxy rate: 0.5420666666666667


In [11]:
df[[
    "content_id",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "refresh_proxy"
]].head(10)

,content_id,impressions_last_30d,impressions_prev_30d,content_age_days,days_since_last_update,trend_direction,refresh_proxy
0,content_304f48230142,578,987,187,20,down,1
1,content_a1fb4e703a9e,2501,5915,445,25,down,1
2,content_9aa793d4d895,2382,6089,141,20,down,1
3,content_331d6c4de07b,3626,4206,463,22,stable,0
4,content_d99b7a2d90ca,4211,6452,263,14,down,1
5,content_d4084a4bc775,617,1009,147,20,down,1
6,content_9a34b442b552,1,13,90,20,down,1
7,content_a63219c6e95a,636,632,445,22,stable,0
8,content_5e6c160719bc,5696,13828,90,20,down,1
9,content_c27558df2b0c,252,356,257,104,down,1


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.